# ContractRisk M2 - Harness de evaluación

Este notebook ejecuta el adapter **RoBERTalex + LoRA de M1** sobre el eval set gold de M2 y produce un scorecard en tres dimensiones: Macro F1, LLM-as-a-judge y utilidad de revisión contractual.

Antes de comenzar, seleccione **Entorno de ejecución -> Cambiar tipo de entorno de ejecución -> GPU T4**. Ejecute todas las celdas en orden. Los resultados no están escritos manualmente: se calculan durante esta corrida.


## 1. Obtener la rama de la Entrega 2

La celda elimina únicamente un clon temporal previo llamado `contractrisk_m2_colab` dentro de `/content` y descarga la rama de trabajo de M2.


In [ ]:
from pathlib import Path
import os
import shutil
import subprocess

REPO_DIR = Path('/content/contractrisk_m2_colab')
os.chdir('/content')
if REPO_DIR.exists():
    shutil.rmtree(REPO_DIR)

subprocess.run([
    'git', 'clone',
    '--branch', 'Entrega/contractrisk-m2',
    '--single-branch',
    'https://github.com/vcastrop/Juan-Camilo-Ramon-Valentina-Castro.git',
    str(REPO_DIR),
], check=True)
os.chdir(REPO_DIR)
print('Directorio de trabajo:', Path.cwd())


## 2. Instalar el entorno reproducible

Las versiones coinciden con M1. PyTorch ya está incluido en Colab y se comprueba que exista una GPU antes de cargar los modelos.


In [ ]:
!pip -q install -r m2/requirements-m2.txt

import torch
import random
import numpy as np

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

assert torch.cuda.is_available(), 'Active una GPU T4 antes de continuar.'
print('PyTorch:', torch.__version__)
print('GPU:', torch.cuda.get_device_name(0))


## 3. Validar el código y el eval set

Primero se ejecutan pruebas sin descargar modelos. Después se validan las 20 filas, las etiquetas permitidas, la unicidad de IDs y la proporción de casos frontera.


In [ ]:
!python m2/test_harness.py

import sys
sys.path.insert(0, str(REPO_DIR / 'm2'))
from harness import load_eval_set

eval_rows = load_eval_set('m2/eval/contractrisk_m2_eval_gold.csv')
frontier_count = sum(row['caso_frontera'] == 'SI' for row in eval_rows)
assert len(eval_rows) == 20
assert frontier_count >= 4
print('Casos gold:', len(eval_rows))
print('Casos frontera:', frontier_count, f'({frontier_count / len(eval_rows):.1%})')


## 4. Ejecutar el harness completo

Esta celda realiza el trabajo principal:

1. carga `BSC-LT/RoBERTalex` y el adapter LoRA incluido en el repositorio;
2. genera las 20 predicciones del baseline M1;
3. libera ese modelo de la GPU;
4. carga `Qwen/Qwen2.5-1.5B-Instruct` como juez abierto;
5. aplica la rúbrica 1-5 a cada caso;
6. calcula las tres dimensiones y la prueba de sesgo de verbosidad.

La primera descarga puede tardar varios minutos.


In [ ]:
import subprocess

subprocess.run([
    sys.executable, 'm2/run_baseline.py',
    '--eval-set', 'm2/eval/contractrisk_m2_eval_gold.csv',
    '--adapter', 'model/contractrisk_robertalex_lora_adapter.zip',
    '--rubric', 'm2/rubrics/judge_rubric_v1_2.json',
    '--output-dir', 'm2/results',
    '--judge-model', 'Qwen/Qwen2.5-1.5B-Instruct',
    '--seed', '42',
], check=True)


## 5. Inspeccionar los resultados

El scorecard contiene exactamente una fila principal por dimensión. Las tablas complementarias permiten revisar clases, casos frontera, fallos del juez y errores individuales.


In [ ]:
import json
import pandas as pd
from IPython.display import display

scorecard = pd.read_csv('m2/results/scorecard_baseline.csv', encoding='utf-8-sig')
predictions = pd.read_csv('m2/results/predictions_baseline.csv', encoding='utf-8-sig')
metrics = json.loads(Path('m2/results/metrics_baseline.json').read_text(encoding='utf-8'))
bias_probe = json.loads(Path('m2/results/judge_bias_probe.json').read_text(encoding='utf-8'))

display(scorecard)
display(pd.DataFrame([metrics]))
display(predictions.loc[~predictions['correct'].astype(bool), [
    'eval_id', 'descripcion_del_proceso', 'expected', 'predicted',
    'caso_frontera', 'judge_score', 'judge_reason', 'domain_utility'
]])
print('Prueba de sesgo:', json.dumps(bias_probe, ensure_ascii=False, indent=2))


## 6. Verificaciones finales

La corrida solo se considera completa si existen las tres dimensiones, las 20 predicciones, puntajes del juez dentro de 1-5 y una salida documentada para la prueba de sesgo.


In [ ]:
assert len(scorecard) == 3
assert set(scorecard['metric']) == {'macro_f1', 'judge_mean_1_5', 'domain_compliance_rate'}
assert len(predictions) == len(eval_rows) == 20
assert predictions['judge_score'].between(1, 5).all()
assert metrics['frontier_count'] == frontier_count
assert bias_probe['bias'] == 'verbosity'
assert metrics['judge_raw_mean_correct'] > metrics['judge_raw_mean_incorrect'], (
    'El juez bruto no distinguió aciertos de errores; no use esta corrida como scorecard final.'
)
print('Verificaciones finales correctas.')


## 7. Descargar la evidencia

El ZIP contiene el scorecard, las predicciones fila por fila, todas las métricas, la metadata de reproducción y la prueba del sesgo del juez. Descárguelo y consérvelo como evidencia de la corrida.


In [ ]:
from google.colab import files

archive = shutil.make_archive('/content/contractrisk_m2_resultados', 'zip', REPO_DIR / 'm2' / 'results')
files.download(archive)
